# Session 8: Data Visualization with Plotly Express
**Python for Data Analysis Workshop | World Development Indicators**

---

## Learning Objectives

By the end of this session you will be able to:
- Understand when to use each chart type
- Create line charts for time-series trends
- Create bar charts for category comparisons
- Create scatter plots for relationships between variables
- Create box plots for distributions
- Create animated charts to show change over time
- Build choropleth maps for geographic data
- Use facets to create small multiples
- Customize titles, colours, and layout

---

## 1. Setup

We use **Plotly Express** (`px`) — a high-level interface for Plotly that creates
beautiful interactive charts with minimal code.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# Optional: set a consistent theme
px_template = "plotly_white"

# Load clean data
try:
    df = pd.read_csv("WDI_clean.csv")
except FileNotFoundError:
    df = pd.read_csv("World_Dev_Indicators.csv")
    df["Region"] = df["Region"].str.strip()

# Rename for brevity
df = df.rename(columns={
    "GDP (current US$)":                          "GDP",
    "GDP per capita (current US$)":               "GDPpc",
    "Life expectancy at birth, total (years)":    "LifeExp",
    "Population, total":                          "Population",
    "Suicide mortality rate (per 100,000 population)": "SuicideRate"
})

print(f"Dataset shape: {df.shape}")
df.head(3)

## 2. Line Charts — Trends Over Time

Line charts are best for showing how a value changes over time.

In [ ]:
# Prepare: average life expectancy by region per year
le_by_region = df.groupby(["Year","Region"])["LifeExp"].mean().reset_index()

# Line chart — one line per region
fig = px.line(
    le_by_region,
    x="Year",
    y="LifeExp",
    color="Region",
    title="Average Life Expectancy by Region (2006–2025)",
    labels={"LifeExp": "Life Expectancy (years)", "Year": "Year"},
    template=px_template
)
fig.update_layout(legend_title_text="Region")
fig.show()

In [ ]:
# Compare specific countries — G7 members
g7 = ["United States","United Kingdom","Germany","France","Japan","Italy","Canada"]
df_g7 = df[df["Country"].isin(g7)].sort_values("Year")

fig2 = px.line(
    df_g7,
    x="Year",
    y="GDPpc",
    color="Country",
    markers=True,       # add dots on each data point
    title="GDP per Capita — G7 Countries (2006–2025)",
    labels={"GDPpc": "GDP per Capita (USD)", "Year": "Year"},
    template=px_template
)
fig2.show()

## 3. Bar Charts — Comparing Categories

Bar charts are best for comparing discrete groups.

In [ ]:
# Average GDP per capita by Income Group in 2020
df_2020 = df[df["Year"] == 2020].copy()
avg_gdppc = df_2020.groupby("Income Group")["GDPpc"].median().reset_index()
avg_gdppc = avg_gdppc.sort_values("GDPpc", ascending=True)

fig3 = px.bar(
    avg_gdppc,
    x="GDPpc",
    y="Income Group",
    orientation="h",           # horizontal bar chart
    title="Median GDP per Capita by Income Group (2020)",
    labels={"GDPpc": "Median GDP per Capita (USD)", "Income Group": ""},
    color="GDPpc",
    color_continuous_scale="Blues",
    template=px_template
)
fig3.show()

In [ ]:
# Top 15 most populous countries in 2020
top_pop = df_2020.nlargest(15, "Population")[["Country","Population","Region"]]
top_pop["Population_B"] = top_pop["Population"] / 1e9

fig4 = px.bar(
    top_pop,
    x="Country",
    y="Population_B",
    color="Region",
    title="Top 15 Most Populous Countries (2020)",
    labels={"Population_B": "Population (Billions)", "Country": ""},
    template=px_template
)
fig4.update_layout(xaxis_tickangle=-40)
fig4.show()

In [ ]:
# Grouped bar: compare regions across income groups
region_income = df_2020.groupby(["Region","Income Group"])["GDPpc"].median().reset_index()

fig5 = px.bar(
    region_income,
    x="Region",
    y="GDPpc",
    color="Income Group",
    barmode="group",
    title="Median GDP per Capita by Region and Income Group (2020)",
    labels={"GDPpc": "Median GDP per Capita (USD)"},
    template=px_template
)
fig5.update_layout(xaxis_tickangle=-30)
fig5.show()

## 4. Scatter Plots — Relationships Between Variables

Scatter plots reveal correlations and outliers between two numeric variables.

In [ ]:
# GDP per capita vs Life Expectancy — 2020
df_2020_clean = df_2020.dropna(subset=["GDPpc","LifeExp","Population"])

fig6 = px.scatter(
    df_2020_clean,
    x="GDPpc",
    y="LifeExp",
    color="Region",
    size="Population",              # bubble size = population
    size_max=50,
    hover_name="Country",           # show country name on hover
    log_x=True,                     # log scale for GDP (better spread)
    title="GDP per Capita vs Life Expectancy (2020)",
    labels={
        "GDPpc":   "GDP per Capita (USD, log scale)",
        "LifeExp": "Life Expectancy (years)"
    },
    template=px_template
)
fig6.show()

In [ ]:
# GDP per capita vs Suicide Rate by Income Group
df_suicide = df_2020.dropna(subset=["GDPpc","SuicideRate"])

fig7 = px.scatter(
    df_suicide,
    x="GDPpc",
    y="SuicideRate",
    color="Income Group",
    hover_name="Country",
    title="GDP per Capita vs Suicide Mortality Rate (2020)",
    labels={
        "GDPpc":       "GDP per Capita (USD)",
        "SuicideRate": "Suicide Rate (per 100k)"
    },
    trendline="ols",                 # add regression line
    template=px_template
)
fig7.show()

## 5. Animated Charts — Change Over Time

Plotly Express supports animation with a single argument.

In [ ]:
# Animated scatter: GDP vs Life Expectancy animated by year
df_anim = df.dropna(subset=["GDPpc","LifeExp","Population"])

fig8 = px.scatter(
    df_anim,
    x="GDPpc",
    y="LifeExp",
    animation_frame="Year",
    animation_group="Country",
    color="Region",
    size="Population",
    size_max=45,
    hover_name="Country",
    log_x=True,
    range_x=[100, 200000],
    range_y=[30, 90],
    title="GDP per Capita vs Life Expectancy Over Time (Animated)",
    labels={
        "GDPpc":   "GDP per Capita (USD, log scale)",
        "LifeExp": "Life Expectancy (years)"
    },
    template=px_template
)
fig8.show()

## 6. Box Plots — Distributions

Box plots show the distribution of a variable within groups.

In [ ]:
# Life expectancy distribution by region
fig9 = px.box(
    df,
    x="Region",
    y="LifeExp",
    color="Region",
    points="outliers",                # show outlier dots
    title="Life Expectancy Distribution by Region (all years)",
    labels={"LifeExp": "Life Expectancy (years)", "Region": ""},
    template=px_template
)
fig9.update_layout(showlegend=False, xaxis_tickangle=-30)
fig9.show()

In [ ]:
# Box plot of GDP per capita by Income Group and recent years
df_recent = df[df["Year"] >= 2015].dropna(subset=["GDPpc"])

fig10 = px.box(
    df_recent,
    x="Income Group",
    y="GDPpc",
    color="Income Group",
    log_y=True,
    title="GDP per Capita Distribution by Income Group (2015–2025)",
    labels={"GDPpc": "GDP per Capita (USD, log scale)"},
    template=px_template
)
fig10.update_layout(showlegend=False)
fig10.show()

## 7. Choropleth Maps — Geographic Data

Choropleth maps colour countries based on a numeric variable.

In [ ]:
# Choropleth: Life expectancy by country in 2020
df_map = df[df["Year"] == 2020].dropna(subset=["LifeExp"])

fig11 = px.choropleth(
    df_map,
    locations="Country Code",
    locationmode="ISO-3",
    color="LifeExp",
    hover_name="Country",
    color_continuous_scale="RdYlGn",  # red = low, green = high
    title="Life Expectancy by Country (2020)",
    labels={"LifeExp": "Life Expectancy (years)"},
    template=px_template
)
fig11.show()

In [ ]:
# Choropleth: GDP per capita (log scale) by country in 2020
df_map2 = df[df["Year"] == 2020].dropna(subset=["GDPpc"])
df_map2 = df_map2.copy()
df_map2["log_GDPpc"] = np.log10(df_map2["GDPpc"] + 1)

fig12 = px.choropleth(
    df_map2,
    locations="Country Code",
    locationmode="ISO-3",
    color="log_GDPpc",
    hover_name="Country",
    hover_data={"GDPpc": ":,.0f", "log_GDPpc": False},
    color_continuous_scale="Viridis",
    title="GDP per Capita by Country (2020, log scale)",
    labels={"log_GDPpc": "Log₁₀(GDP per Capita)"},
    template=px_template
)
fig12.show()

## 8. Faceted Charts — Small Multiples

Facets split one chart into a grid by a categorical variable.
This lets you compare patterns across groups side by side.

In [ ]:
# Facet line chart: GDP per capita over time, one panel per region
avg_gdppc_region = df.groupby(["Year","Region"])["GDPpc"].mean().reset_index()

fig13 = px.line(
    avg_gdppc_region,
    x="Year",
    y="GDPpc",
    facet_col="Region",
    facet_col_wrap=3,           # 3 panels per row
    title="GDP per Capita Trend by Region",
    labels={"GDPpc": "Mean GDP per Capita (USD)"},
    template=px_template
)
fig13.update_layout(height=700)
fig13.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig13.show()

In [ ]:
# Facet scatter: GDP vs Life Expectancy faceted by Income Group
df_facet = df[df["Year"] == 2020].dropna(subset=["GDPpc","LifeExp"])

fig14 = px.scatter(
    df_facet,
    x="GDPpc",
    y="LifeExp",
    facet_row="Income Group",
    color="Region",
    hover_name="Country",
    log_x=True,
    title="GDP per Capita vs Life Expectancy by Income Group (2020)",
    labels={"GDPpc": "GDP per Capita (USD, log)", "LifeExp": "Life Expectancy"},
    template=px_template
)
fig14.update_layout(height=900)
fig14.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig14.show()

## 9. Customisation Tips

Polish your charts with titles, colours, annotations, and layout tweaks.

In [ ]:
# Fully customised bar chart
df_2020_region = df[df["Year"]==2020].groupby("Region")["LifeExp"].mean().reset_index()
df_2020_region = df_2020_region.sort_values("LifeExp")

fig15 = px.bar(
    df_2020_region,
    x="LifeExp",
    y="Region",
    orientation="h",
    color="LifeExp",
    color_continuous_scale="RdYlGn",
    text="LifeExp",             # show values on bars
    title="Average Life Expectancy by World Region (2020)",
    labels={"LifeExp": "Life Expectancy (years)", "Region": ""},
    template=px_template
)
# Text formatting
fig15.update_traces(texttemplate="%{text:.1f}", textposition="outside")

# Layout adjustments
fig15.update_layout(
    title_font_size=16,
    title_x=0.5,                # centre title
    coloraxis_showscale=False,  # hide colour bar
    xaxis_range=[50, 85],
    margin=dict(l=180, r=40, t=60, b=40),
    height=420
)
fig15.show()

In [ ]:
# Saving charts to HTML (interactive) and PNG (static)
# Save as interactive HTML
fig15.write_html("life_expectancy_by_region_2020.html")
print("Saved: life_expectancy_by_region_2020.html (interactive)")

# Save as static PNG — requires: pip install kaleido
try:
    fig15.write_image("life_expectancy_by_region_2020.png", width=900, height=450)
    print("Saved: life_expectancy_by_region_2020.png (static)")
except Exception as e:
    print(f"PNG export requires kaleido: pip install kaleido\nError: {e}")

## 10. Choosing the Right Chart

| Goal | Chart Type |
|------|------------|
| Show trend over time | Line chart |
| Compare categories | Bar chart (vertical or horizontal) |
| Show relationship between 2 numbers | Scatter plot |
| Show distribution within groups | Box plot |
| Show geographic patterns | Choropleth map |
| Compare many groups side by side | Faceted chart |
| Show composition | Stacked bar / Pie (use sparingly) |
| Show change over time interactively | Animated scatter |

---

## 11. Summary Cheat Sheet

| Chart | Function | Key Args |
|-------|----------|----------|
| Line | `px.line()` | x, y, color |
| Bar | `px.bar()` | x, y, color, barmode, orientation |
| Scatter | `px.scatter()` | x, y, color, size, hover_name, log_x |
| Animated scatter | `px.scatter()` | + animation_frame, animation_group |
| Box plot | `px.box()` | x, y, color, points |
| Choropleth | `px.choropleth()` | locations, locationmode, color |
| Facet | any px + | facet_col or facet_row |
| Save HTML | `fig.write_html('file.html')` | — |
| Save PNG | `fig.write_image('file.png')` | width, height |

---

## 12. Exercises

1. Create a line chart showing **global average GDP per capita** from 2006 to 2025 (average across all countries per year).
2. Create a horizontal bar chart showing the **top 10 countries by life expectancy in 2020**, coloured by Region.
3. Create a scatter plot of **Population vs GDP** (in trillions) for the year 2020, with country name on hover and colour by Income Group.
4. Create a faceted line chart showing **life expectancy trends** for South Asian countries — one panel per country.
5. **Bonus:** Create a choropleth animation showing how **suicide rates** changed from 2006 to 2019 (use `animation_frame="Year"` in `px.choropleth()`).

In [ ]:
# Your answers here:
# Exercise 1 — Global average GDP per capita over time

# Exercise 2 — Top 10 life expectancy bar chart (2020)

# Exercise 3 — Population vs GDP scatter (2020)

# Exercise 4 — South Asia life expectancy faceted line chart

# Exercise 5 (bonus) — Animated suicide rate choropleth
